In [1]:
import os
import numpy as np
import pandas as pd

from ortools.sat.python import cp_model

BASE_DIR = r"C:\Users\diksh\OneDrive\Desktop\RailWise"
DATA_DIR = os.path.join(BASE_DIR, "SIH_26027_Final_Dataset")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
MODEL_DIR = os.path.join(BASE_DIR, "models")

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

BASE_DIR: C:\Users\diksh\OneDrive\Desktop\RailWise
DATA_DIR: C:\Users\diksh\OneDrive\Desktop\RailWise\SIH_26027_Final_Dataset
OUTPUT_DIR: C:\Users\diksh\OneDrive\Desktop\RailWise\outputs


In [2]:
maintenance = pd.read_csv(
    os.path.join(DATA_DIR, "unified_maintenance.csv")
)

block_requests = pd.read_csv(
    os.path.join(DATA_DIR, "block_requests.csv")
)

coa = pd.read_csv(
    os.path.join(DATA_DIR, "coa_block_availability.csv")
)

movements = pd.read_csv(
    os.path.join(DATA_DIR, "train_movement_windows.csv")
)

train_schedule = pd.read_csv(
    os.path.join(DATA_DIR, "train_schedule.csv")
)

goods_forecast = pd.read_csv(
    os.path.join(DATA_DIR, "goods_train_forecast.csv")
)

historical = pd.read_csv(
    os.path.join(DATA_DIR, "historical_block_plans.csv")
)

corridors = pd.read_csv(
    os.path.join(DATA_DIR, "corridors.csv")
)

print("Maintenance:", maintenance.shape)
print("Block requests:", block_requests.shape)
print("COA:", coa.shape)
print("Train movements:", movements.shape)
print("Train schedule:", train_schedule.shape)
print("Goods forecast:", goods_forecast.shape)
print("Historical:", historical.shape)
print("Corridors:", corridors.shape)

Maintenance: (14400, 21)
Block requests: (6000, 8)
COA: (1500, 8)
Train movements: (15058, 11)
Train schedule: (26736, 16)
Goods forecast: (3000, 6)
Historical: (4000, 10)
Corridors: (100, 15)


In [3]:
import joblib

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "xgboost_maintenance_priority_model.pkl"
)

FEATURE_PATH = os.path.join(
    MODEL_DIR,
    "model_features.pkl"
)

xgb_model = joblib.load(MODEL_PATH)
model_features = joblib.load(FEATURE_PATH)

print("XGBoost model loaded")
print("Number of model features:", len(model_features))

XGBoost model loaded
Number of model features: 140


In [4]:
features = [
    "department",
    "asset_type",
    "corridor_id",
    "location_km",
    "criticality_1_5",
    "safety_risk_1_5",
    "operational_impact_1_5",
    "overdue_days",
    "estimated_duration_min",
    "required_team_size",
    "possession_required",
    "maintenance_type",
    "severity"
]

prediction_df = maintenance.copy()

X = prediction_df[features].copy()

numeric_features = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns

for col in numeric_features:
    X[col] = X[col].fillna(X[col].median())

for col in categorical_features:
    X[col] = X[col].fillna("Unknown").astype(str)

X = pd.get_dummies(
    X,
    columns=categorical_features,
    drop_first=True
)

# Match training feature structure exactly
X = X.reindex(
    columns=model_features,
    fill_value=0
)

prediction_df["predicted_priority"] = xgb_model.predict(X)

prediction_df["priority_category"] = pd.cut(
    prediction_df["predicted_priority"],
    bins=[-np.inf, 40, 60, 80, np.inf],
    labels=["Low", "Medium", "High", "Critical"]
)

print(
    prediction_df[
        ["task_id", "corridor_id",
         "predicted_priority", "priority_category"]
    ].head()
)

# ------------------------------------------------------------
# Save AI risk predictions
# ------------------------------------------------------------
asset_risk_predictions = prediction_df[
    [
        "task_id",
        "asset_id",
        "asset_type",
        "department",
        "corridor_id",
        "location_km",
        "criticality_1_5",
        "safety_risk_1_5",
        "operational_impact_1_5",
        "overdue_days",
        "severity",
        "predicted_priority",
        "priority_category"
    ]
].copy()

asset_risk_predictions.to_csv(
    os.path.join(OUTPUT_DIR, "asset_risk_predictions.csv"),
    index=False
)

print(
    "Saved asset_risk_predictions.csv:",
    len(asset_risk_predictions)
)

# ------------------------------------------------------------
# Save AI risk predictions
# ------------------------------------------------------------
asset_risk_predictions = prediction_df[
    [
        "task_id",
        "asset_id",
        "asset_type",
        "department",
        "corridor_id",
        "location_km",
        "criticality_1_5",
        "safety_risk_1_5",
        "operational_impact_1_5",
        "overdue_days",
        "severity",
        "predicted_priority",
        "priority_category"
    ]
].copy()

asset_risk_predictions.to_csv(
    os.path.join(OUTPUT_DIR, "asset_risk_predictions.csv"),
    index=False
)

print(
    "Saved asset_risk_predictions.csv:",
    len(asset_risk_predictions)
)


     task_id corridor_id  predicted_priority priority_category
0  TMS000001      C00069           40.444424            Medium
1  TMS000002      C00033           24.116785               Low
2  TMS000003      C00078           54.487446            Medium
3  TMS000004      C00002           52.910378            Medium
4  TMS000005      C00078           67.689354              High
Saved asset_risk_predictions.csv: 14400
Saved asset_risk_predictions.csv: 14400


In [5]:
task_info = prediction_df[
    [
        "task_id",
        "asset_id",
        "asset_type",
        "department",
        "corridor_id",
        "location_km",
        "severity",
        "criticality_1_5",
        "safety_risk_1_5",
        "operational_impact_1_5",
        "overdue_days",
        "estimated_duration_min",
        "required_team_size",
        "possession_required",
        "maintenance_type",
        "status",
        "predicted_priority",
        "priority_category"
    ]
].copy()

requests = block_requests.merge(
    task_info,
    on="task_id",
    how="left",
    suffixes=("", "_task")
)

print("Requests with task information:", requests.shape)

requests[
    [
        "block_request_id",
        "task_id",
        "corridor_id",
        "requested_duration_min",
        "requested_date",
        "predicted_priority",
        "priority_category"
    ]
].head()

Requests with task information: (6000, 25)


,block_request_id,task_id,corridor_id,requested_duration_min,requested_date,predicted_priority,priority_category
0,BR000001,TMS000001,C00069,75,2025-06-03,40.444424,Medium
1,BR000002,TMS000003,C00078,91,2025-06-21,54.487446,Medium
2,BR000003,TMS000004,C00002,114,2025-06-08,52.910378,Medium
3,BR000004,TMS000005,C00078,78,2025-06-04,67.689354,High
4,BR000005,TMS000006,C00041,115,2025-06-14,27.490358,Low


In [6]:
available_coa = coa[
    coa["availability_status"].astype(str).str.lower() == "available"
].copy()

available_coa["block_date"] = pd.to_datetime(
    available_coa["block_date"]
)

available_coa["window_start_min"] = (
    pd.to_datetime(
        available_coa["window_start"],
        format="%H:%M"
    ).dt.hour * 60
    +
    pd.to_datetime(
        available_coa["window_start"],
        format="%H:%M"
    ).dt.minute
)

available_coa["window_end_min"] = (
    pd.to_datetime(
        available_coa["window_end"],
        format="%H:%M"
    ).dt.hour * 60
    +
    pd.to_datetime(
        available_coa["window_end"],
        format="%H:%M"
    ).dt.minute
)

print("Available COA windows:", len(available_coa))

Available COA windows: 1366


In [7]:
candidate_rows = []

for _, req in requests.iterrows():

    if pd.isna(req["requested_duration_min"]):
        continue

    duration = int(req["requested_duration_min"])

    req_date = pd.to_datetime(req["requested_date"])
    corridor = req["corridor_id"]

    matching_windows = available_coa[
        (available_coa["corridor_id"] == corridor) &
        (available_coa["block_date"] == req_date) &
        (available_coa["available_duration_min"] >= duration)
    ]

    for _, window in matching_windows.iterrows():

        start_min = int(window["window_start_min"])
        end_min = start_min + duration

        if end_min <= int(window["window_end_min"]):

            candidate_rows.append({
                "block_request_id": req["block_request_id"],
                "task_id": req["task_id"],
                "asset_id": req["asset_id"],
                "corridor_id": corridor,
                "block_window_id": window["block_window_id"],
                "block_date": req_date.date(),
                "start_min": start_min,
                "end_min": end_min,
                "duration_min": duration,
                "predicted_priority": req["predicted_priority"],
                "priority_category": req["priority_category"],
                "department": req["department"],
                "possession_required": req["possession_required"]
            })

candidate_blocks = pd.DataFrame(candidate_rows)

print("Candidate blocks:", candidate_blocks.shape)

candidate_blocks.head()

Candidate blocks: (859, 13)


,block_request_id,task_id,asset_id,corridor_id,block_window_id,block_date,start_min,end_min,duration_min,predicted_priority,priority_category,department,possession_required
0,BR000002,TMS000003,AST000826,C00078,COA000408,2025-06-21,155,246,91,54.487446,Medium,Engineering,No
1,BR000004,TMS000005,AST001012,C00078,COA000722,2025-06-04,988,1066,78,67.689354,High,Engineering,Yes
2,BR000007,TMS000008,AST004121,C00094,COA001390,2025-06-10,1057,1203,146,43.130348,Medium,Engineering,Yes
3,BR000019,TMS000021,AST003486,C00003,COA000390,2025-06-28,206,249,43,62.854858,High,Engineering,Yes
4,BR000024,TMS000027,AST003894,C00093,COA000139,2025-06-25,608,644,36,49.294453,Medium,Engineering,Yes


In [8]:
def time_to_minutes(value):
    if pd.isna(value):
        return np.nan

    value = str(value)

    hour, minute = map(int, value.split(":")[:2])

    return hour * 60 + minute


movements["arrival_min"] = movements["arrival_time"].apply(
    time_to_minutes
)

movements["next_arrival_min"] = movements["next_arrival_time"].apply(
    time_to_minutes
)

movements["movement_date"] = pd.to_datetime(
    movements["movement_date"]
)

print(
    movements[
        [
            "train_no",
            "corridor_id",
            "movement_date",
            "arrival_min",
            "next_arrival_min",
            "conflict_buffer_min"
        ]
    ].head()
)

   train_no corridor_id movement_date  arrival_min  next_arrival_min  \
0     12014      C00016    2025-06-02         1122              1177   
1     12014      C00098    2025-06-03         1209              1218   
2     12014      C00022    2025-06-30         1267              1417   
3     12014      C00092    2025-06-26           19                95   
4     12014      C00053    2025-06-23          145               237   

   conflict_buffer_min  
0                   10  
1                    9  
2                    7  
3                   13  
4                    6  


In [9]:
conflict_rows = []

for idx, candidate in candidate_blocks.iterrows():

    relevant = movements[
        (movements["corridor_id"] == candidate["corridor_id"]) &
        (movements["movement_date"] == pd.Timestamp(candidate["block_date"]))
    ]

    for _, movement in relevant.iterrows():

        buffer = int(
            movement["conflict_buffer_min"]
            if not pd.isna(movement["conflict_buffer_min"])
            else 0
        )

        movement_start = int(movement["arrival_min"])
        movement_end = int(movement["next_arrival_min"])

        block_start = int(candidate["start_min"])
        block_end = int(candidate["end_min"])

        buffered_start = movement_start - buffer
        buffered_end = movement_end + buffer

        overlap = (
            buffered_start < block_end
            and buffered_end > block_start
        )

        if overlap:

            overlap_start = max(
                block_start,
                buffered_start
            )

            overlap_end = min(
                block_end,
                buffered_end
            )

            conflict_duration = max(
                0,
                overlap_end - overlap_start
            )

            conflict_rows.append({
                "candidate_id": idx,
                "block_request_id": candidate["block_request_id"],
                "task_id": candidate["task_id"],
                "block_window_id": candidate["block_window_id"],
                "corridor_id": candidate["corridor_id"],
                "train_no": movement["train_no"],
                "movement_id": movement["movement_id"],
                "movement_type": movement["movement_type"],
                "movement_start": movement["arrival_time"],
                "movement_end": movement["next_arrival_time"],
                "conflict_buffer_min": buffer,
                "conflict_duration_min": conflict_duration
            })

conflict_analysis = pd.DataFrame(conflict_rows)

print("Conflicts detected:", len(conflict_analysis))

conflict_analysis.head()

Conflicts detected: 743


,candidate_id,block_request_id,task_id,block_window_id,corridor_id,train_no,movement_id,movement_type,movement_start,movement_end,conflict_buffer_min,conflict_duration_min
0,0,BR000002,TMS000003,COA000408,C00078,56202,MW009728,Train Movement,00:56,04:11,11,91
1,2,BR000007,TMS000008,COA001390,C00094,14894,MW004191,Train Movement,19:16,22:09,9,56
2,3,BR000019,TMS000021,COA000390,C00003,57042,MW010012,Train Movement,04:09,06:20,14,14
3,4,BR000024,TMS000027,COA000139,C00093,65891,MW011329,Train Movement,10:31,12:39,8,21
4,6,BR000043,TMS000053,COA000527,C00086,13394,MW001893,Train Movement,06:22,09:21,12,50


In [10]:
if len(conflict_analysis) > 0:

    conflict_summary = (
        conflict_analysis
        .groupby("candidate_id")
        .agg(
            train_conflicts=("train_no", "nunique"),
            total_conflict_duration_min=(
                "conflict_duration_min",
                "sum"
            )
        )
        .reset_index()
    )

else:

    conflict_summary = pd.DataFrame(
        columns=[
            "candidate_id",
            "train_conflicts",
            "total_conflict_duration_min"
        ]
    )


candidate_blocks = candidate_blocks.reset_index(drop=True)

candidate_blocks["candidate_id"] = candidate_blocks.index

candidate_blocks = candidate_blocks.merge(
    conflict_summary,
    on="candidate_id",
    how="left"
)

candidate_blocks["train_conflicts"] = (
    candidate_blocks["train_conflicts"]
    .fillna(0)
    .astype(int)
)

candidate_blocks["total_conflict_duration_min"] = (
    candidate_blocks["total_conflict_duration_min"]
    .fillna(0)
)

print(
    candidate_blocks[
        [
            "candidate_id",
            "task_id",
            "corridor_id",
            "block_window_id",
            "duration_min",
            "predicted_priority",
            "train_conflicts",
            "total_conflict_duration_min"
        ]
    ].head(10)
)

   candidate_id    task_id corridor_id block_window_id  duration_min  \
0             0  TMS000003      C00078       COA000408            91   
1             1  TMS000005      C00078       COA000722            78   
2             2  TMS000008      C00094       COA001390           146   
3             3  TMS000021      C00003       COA000390            43   
4             4  TMS000027      C00093       COA000139            36   
5             5  TMS000028      C00076       COA001405           110   
6             6  TMS000053      C00086       COA000527           129   
7             7  TMS000055      C00066       COA000597            33   
8             8  TMS000065      C00001       COA001168            63   
9             9  TMS000065      C00001       COA001199            63   

   predicted_priority  train_conflicts  total_conflict_duration_min  
0           54.487446                1                         91.0  
1           67.689354                0                          0.0

In [11]:
goods_forecast["forecast_date"] = pd.to_datetime(
    goods_forecast["forecast_date"]
)

candidate_blocks["block_date"] = pd.to_datetime(
    candidate_blocks["block_date"]
)

candidate_blocks = candidate_blocks.merge(
    goods_forecast[
        [
            "corridor_id",
            "forecast_date",
            "forecast_goods_trains",
            "peak_period",
            "confidence_pct"
        ]
    ],
    left_on=["corridor_id", "block_date"],
    right_on=["corridor_id", "forecast_date"],
    how="left"
)

candidate_blocks["forecast_goods_trains"] = (
    candidate_blocks["forecast_goods_trains"]
    .fillna(0)
)

candidate_blocks["confidence_pct"] = (
    candidate_blocks["confidence_pct"]
    .fillna(0)
)

candidate_blocks.head()

,block_request_id,task_id,asset_id,corridor_id,block_window_id,block_date,start_min,end_min,duration_min,predicted_priority,priority_category,department,possession_required,candidate_id,train_conflicts,total_conflict_duration_min,forecast_date,forecast_goods_trains,peak_period,confidence_pct
0,BR000002,TMS000003,AST000826,C00078,COA000408,2025-06-21,155,246,91,54.487446,Medium,Engineering,No,0,1,91.0,NaT,0.0,NaN,0.0
1,BR000004,TMS000005,AST001012,C00078,COA000722,2025-06-04,988,1066,78,67.689354,High,Engineering,Yes,1,0,0.0,2025-06-04,7.0,Normal,85.0
2,BR000004,TMS000005,AST001012,C00078,COA000722,2025-06-04,988,1066,78,67.689354,High,Engineering,Yes,1,0,0.0,2025-06-04,3.0,Off-Peak,82.0
3,BR000007,TMS000008,AST004121,C00094,COA001390,2025-06-10,1057,1203,146,43.130348,Medium,Engineering,Yes,2,1,56.0,2025-06-10,3.0,Off-Peak,86.0
4,BR000007,TMS000008,AST004121,C00094,COA001390,2025-06-10,1057,1203,146,43.130348,Medium,Engineering,Yes,2,1,56.0,2025-06-10,4.0,Off-Peak,76.0


In [12]:
candidate_blocks.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "candidate_blocks.csv"
    ),
    index=False
)

conflict_analysis.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "conflict_analysis.csv"
    ),
    index=False
)

print("Candidate and conflict files saved.")

Candidate and conflict files saved.


In [13]:
print("Candidates:", len(candidate_blocks))
print("Conflicts:", len(conflict_analysis))

print(
    candidate_blocks[
        [
            "candidate_id",
            "task_id",
            "corridor_id",
            "block_window_id",
            "duration_min",
            "predicted_priority",
            "train_conflicts",
            "total_conflict_duration_min"
        ]
    ].head(20).to_string(index=False)
)

Candidates: 1204
Conflicts: 743
 candidate_id   task_id corridor_id block_window_id  duration_min  predicted_priority  train_conflicts  total_conflict_duration_min
            0 TMS000003      C00078       COA000408            91           54.487446                1                         91.0
            1 TMS000005      C00078       COA000722            78           67.689354                0                          0.0
            1 TMS000005      C00078       COA000722            78           67.689354                0                          0.0
            2 TMS000008      C00094       COA001390           146           43.130348                1                         56.0
            2 TMS000008      C00094       COA001390           146           43.130348                1                         56.0
            2 TMS000008      C00094       COA001390           146           43.130348                1                         56.0
            3 TMS000021      C00003       CO

In [14]:
# Work on a clean copy
opt_candidates = candidate_blocks.copy()

# Candidate must have a valid task
opt_candidates = opt_candidates[
    opt_candidates["task_id"].notna()
].copy()

# Give every candidate a unique integer index
opt_candidates = opt_candidates.reset_index(drop=True)
opt_candidates["candidate_id"] = opt_candidates.index

print("Optimization candidates:", len(opt_candidates))

print(
    opt_candidates[
        [
            "candidate_id",
            "task_id",
            "block_request_id",
            "corridor_id",
            "block_window_id",
            "duration_min",
            "predicted_priority",
            "train_conflicts",
            "total_conflict_duration_min"
        ]
    ].head(10)
)

Optimization candidates: 1204
   candidate_id    task_id block_request_id corridor_id block_window_id  \
0             0  TMS000003         BR000002      C00078       COA000408   
1             1  TMS000005         BR000004      C00078       COA000722   
2             2  TMS000005         BR000004      C00078       COA000722   
3             3  TMS000008         BR000007      C00094       COA001390   
4             4  TMS000008         BR000007      C00094       COA001390   
5             5  TMS000008         BR000007      C00094       COA001390   
6             6  TMS000021         BR000019      C00003       COA000390   
7             7  TMS000027         BR000024      C00093       COA000139   
8             8  TMS000028         BR000025      C00076       COA001405   
9             9  TMS000053         BR000043      C00086       COA000527   

   duration_min  predicted_priority  train_conflicts  \
0            91           54.487446                1   
1            78           67.689

In [15]:
print("Total candidates:", len(opt_candidates))

print(
    "Tasks having at least one candidate:",
    opt_candidates["task_id"].nunique()
)

print(
    "Tasks in block requests:",
    requests["task_id"].nunique()
)

print(
    "Candidates with zero train conflicts:",
    (opt_candidates["train_conflicts"] == 0).sum()
)

print(
    "Candidates with train conflicts:",
    (opt_candidates["train_conflicts"] > 0).sum()
)

Total candidates: 1204
Tasks having at least one candidate: 749
Tasks in block requests: 6000
Candidates with zero train conflicts: 557
Candidates with train conflicts: 647


In [16]:
model = cp_model.CpModel()

candidate_vars = {}

for idx in opt_candidates.index:
    candidate_vars[idx] = model.NewBoolVar(
        f"select_{idx}"
    )

print("Binary decision variables:", len(candidate_vars))

Binary decision variables: 1204


In [17]:
task_groups = opt_candidates.groupby("task_id")

for task_id, group in task_groups:

    indices = group.index.tolist()

    model.Add(
        sum(candidate_vars[i] for i in indices) <= 1
    )

print("Task assignment constraint added.")

Task assignment constraint added.


In [18]:
# Group candidates by actual COA window
window_groups = opt_candidates.groupby("block_window_id")

overlap_constraints = 0

for window_id, group in window_groups:

    indices = group.index.tolist()

    for i in range(len(indices)):
        for j in range(i + 1, len(indices)):

            a = indices[i]
            b = indices[j]

            row_a = opt_candidates.loc[a]
            row_b = opt_candidates.loc[b]

            # Check whether the two candidate intervals overlap
            overlap = (
                row_a["start_min"] < row_b["end_min"]
                and
                row_b["start_min"] < row_a["end_min"]
            )

            if overlap:

                model.Add(
                    candidate_vars[a] +
                    candidate_vars[b]
                    <= 1
                )

                overlap_constraints += 1

print(
    "Overlapping block constraints:",
    overlap_constraints
)

Overlapping block constraints: 1495


In [19]:
objective_terms = []

for idx, row in opt_candidates.iterrows():

    priority = float(row["predicted_priority"])

    conflicts = int(row["train_conflicts"])

    conflict_duration = float(
        row["total_conflict_duration_min"]
    )

    goods_trains = float(
        row["forecast_goods_trains"]
    )

    duration = float(row["duration_min"])

    # Maintenance value
    maintenance_value = priority * 100

    # Operational disruption penalties
    conflict_penalty = conflicts * 5000

    duration_penalty = conflict_duration * 100

    goods_penalty = goods_trains * 20

    # Small preference for shorter blocks
    duration_cost = duration * 1

    score = (
        maintenance_value
        - conflict_penalty
        - duration_penalty
        - goods_penalty
        - duration_cost
    )

    objective_terms.append(
        int(round(score)) * candidate_vars[idx]
    )

model.Maximize(
    sum(objective_terms)
)

print("Optimization objective created.")

Optimization objective created.


In [20]:
solver = cp_model.CpSolver()

solver.parameters.max_time_in_seconds = 30
solver.parameters.num_search_workers = 8

status = solver.Solve(model)

print("Solver status:", solver.StatusName(status))

Solver status: OPTIMAL


In [21]:
selected_rows = []

for idx, row in opt_candidates.iterrows():

    if solver.Value(candidate_vars[idx]) == 1:

        selected_rows.append(
            row.to_dict()
        )

optimized_blocks = pd.DataFrame(selected_rows)

print(
    "Optimized blocks:",
    len(optimized_blocks)
)

if len(optimized_blocks) > 0:

    display(
        optimized_blocks[
            [
                "task_id",
                "block_request_id",
                "asset_id",
                "corridor_id",
                "block_window_id",
                "block_date",
                "start_min",
                "end_min",
                "duration_min",
                "predicted_priority",
                "priority_category",
                "train_conflicts",
                "total_conflict_duration_min",
                "forecast_goods_trains"
            ]
        ].head(20)
    )

Optimized blocks: 258


,task_id,block_request_id,asset_id,corridor_id,block_window_id,block_date,start_min,end_min,duration_min,predicted_priority,priority_category,train_conflicts,total_conflict_duration_min,forecast_goods_trains
0,TMS000005,BR000004,AST001012,C00078,COA000722,2025-06-04,988,1066,78,67.689354,High,0,0.0,3.0
1,TMS000028,BR000025,AST001133,C00076,COA001405,2025-06-02,1146,1256,110,59.352386,Medium,0,0.0,0.0
2,TMS000055,BR000044,AST004019,C00066,COA000597,2025-06-25,68,101,33,26.743483,Low,0,0.0,2.0
3,TMS000065,BR000051,AST000013,C00001,COA001199,2025-06-02,803,866,63,77.893135,High,0,0.0,0.0
4,TMS000076,BR000062,AST002317,C00046,COA000796,2025-06-15,359,478,119,54.495583,Medium,0,0.0,4.0
5,TMS000086,BR000070,AST000036,C00017,COA000826,2025-06-20,186,239,53,74.352638,High,0,0.0,0.0
6,TMS000098,BR000081,AST003042,C00006,COA001192,2025-06-01,40,74,34,37.731422,Low,0,0.0,3.0
7,TMS000103,BR000086,AST004028,C00050,COA001121,2025-06-19,1043,1090,47,30.329281,Low,0,0.0,5.0
8,TMS000132,BR000110,AST003824,C00008,COA001474,2025-06-10,1141,1214,73,27.067982,Low,0,0.0,0.0
9,TMS000141,BR000115,AST001048,C00093,COA001291,2025-06-13,768,842,74,44.866592,Medium,0,0.0,4.0


In [22]:
def minutes_to_time(minutes):
    minutes = int(minutes) % (24 * 60)

    hour = minutes // 60
    minute = minutes % 60

    return f"{hour:02d}:{minute:02d}"


if len(optimized_blocks) > 0:

    optimized_blocks["recommended_start"] = (
        optimized_blocks["start_min"]
        .apply(minutes_to_time)
    )

    optimized_blocks["recommended_end"] = (
        optimized_blocks["end_min"]
        .apply(minutes_to_time)
    )

optimized_blocks.head()

,block_request_id,task_id,asset_id,corridor_id,block_window_id,block_date,start_min,end_min,duration_min,predicted_priority,...,possession_required,candidate_id,train_conflicts,total_conflict_duration_min,forecast_date,forecast_goods_trains,peak_period,confidence_pct,recommended_start,recommended_end
0,BR000004,TMS000005,AST001012,C00078,COA000722,2025-06-04,988,1066,78,67.689354,...,Yes,2,0,0.0,2025-06-04,3.0,Off-Peak,82.0,16:28,17:46
1,BR000025,TMS000028,AST001133,C00076,COA001405,2025-06-02,1146,1256,110,59.352386,...,Yes,8,0,0.0,NaT,0.0,NaN,0.0,19:06,20:56
2,BR000044,TMS000055,AST004019,C00066,COA000597,2025-06-25,68,101,33,26.743483,...,Yes,11,0,0.0,2025-06-25,2.0,Normal,89.0,01:08,01:41
3,BR000051,TMS000065,AST000013,C00001,COA001199,2025-06-02,803,866,63,77.893135,...,Yes,13,0,0.0,NaT,0.0,NaN,0.0,13:23,14:26
4,BR000062,TMS000076,AST002317,C00046,COA000796,2025-06-15,359,478,119,54.495583,...,Yes,16,0,0.0,2025-06-15,4.0,Normal,73.0,05:59,07:58


In [23]:
# ============================================================
# RAILWISE — FINAL OPTIMIZED BLOCK TABLE
# ============================================================

if len(optimized_blocks) > 0:

    final_optimized = optimized_blocks.copy()

    # Selected by the optimizer = feasible selected recommendation
    final_optimized["feasibility"] = "FEASIBLE"

    final_optimized["reason_for_recommendation"] = (
        "Selected by the global CP-SAT optimizer based on "
        "maintenance priority, train conflicts, conflict duration, "
        "goods-train activity and block constraints."
    )

    final_optimized = final_optimized[
        [
            "block_request_id",
            "task_id",
            "asset_id",
            "department",
            "corridor_id",
            "block_window_id",
            "block_date",
            "predicted_priority",
            "priority_category",
            "recommended_start",
            "recommended_end",
            "duration_min",
            "train_conflicts",
            "total_conflict_duration_min",
            "forecast_goods_trains",
            "feasibility",
            "reason_for_recommendation"
        ]
    ].copy()

    final_optimized = final_optimized.rename(
        columns={
            "block_date": "recommended_date",
            "duration_min": "recommended_duration_min",
            "train_conflicts": "train_conflict_count",
            "total_conflict_duration_min": "conflict_duration_min"
        }
    )

else:

    final_optimized = pd.DataFrame(
        columns=[
            "block_request_id",
            "task_id",
            "asset_id",
            "department",
            "corridor_id",
            "block_window_id",
            "recommended_date",
            "predicted_priority",
            "priority_category",
            "recommended_start",
            "recommended_end",
            "recommended_duration_min",
            "train_conflict_count",
            "conflict_duration_min",
            "forecast_goods_trains",
            "feasibility",
            "reason_for_recommendation"
        ]
    )

print("Final optimized columns:")
print(final_optimized.columns.tolist())

display(final_optimized.head())


Final optimized columns:
['block_request_id', 'task_id', 'asset_id', 'department', 'corridor_id', 'block_window_id', 'recommended_date', 'predicted_priority', 'priority_category', 'recommended_start', 'recommended_end', 'recommended_duration_min', 'train_conflict_count', 'conflict_duration_min', 'forecast_goods_trains', 'feasibility', 'reason_for_recommendation']


,block_request_id,task_id,asset_id,department,corridor_id,block_window_id,recommended_date,predicted_priority,priority_category,recommended_start,recommended_end,recommended_duration_min,train_conflict_count,conflict_duration_min,forecast_goods_trains,feasibility,reason_for_recommendation
0,BR000004,TMS000005,AST001012,Engineering,C00078,COA000722,2025-06-04,67.689354,High,16:28,17:46,78,0,0.0,3.0,FEASIBLE,Selected by the global CP-SAT optimizer based ...
1,BR000025,TMS000028,AST001133,Engineering,C00076,COA001405,2025-06-02,59.352386,Medium,19:06,20:56,110,0,0.0,0.0,FEASIBLE,Selected by the global CP-SAT optimizer based ...
2,BR000044,TMS000055,AST004019,Engineering,C00066,COA000597,2025-06-25,26.743483,Low,01:08,01:41,33,0,0.0,2.0,FEASIBLE,Selected by the global CP-SAT optimizer based ...
3,BR000051,TMS000065,AST000013,Engineering,C00001,COA001199,2025-06-02,77.893135,High,13:23,14:26,63,0,0.0,0.0,FEASIBLE,Selected by the global CP-SAT optimizer based ...
4,BR000062,TMS000076,AST002317,Engineering,C00046,COA000796,2025-06-15,54.495583,Medium,05:59,07:58,119,0,0.0,4.0,FEASIBLE,Selected by the global CP-SAT optimizer based ...


In [24]:
optimized_path = os.path.join(
    OUTPUT_DIR,
    "optimized_blocks.csv"
)

final_optimized.to_csv(
    optimized_path,
    index=False
)

print("Saved:")
print(optimized_path)

Saved:
C:\Users\diksh\OneDrive\Desktop\RailWise\outputs\optimized_blocks.csv


In [26]:
# ============================================================
# RAILWISE — OPTIMIZATION SUMMARY
# ============================================================

print("=" * 70)
print("RAILWISE — OPTIMIZATION SUMMARY")
print("=" * 70)

print(f"Maintenance tasks          : {len(maintenance):,}")
print(f"Block requests             : {len(block_requests):,}")
print(f"Available COA windows      : {len(available_coa):,}")
print(f"Candidate blocks           : {len(opt_candidates):,}")
print(f"Tasks with candidates      : {opt_candidates['task_id'].nunique():,}")
print(f"Optimized blocks           : {len(final_optimized):,}")

# Priority counts
critical_count = (
    final_optimized["priority_category"] == "Critical"
).sum()

high_count = (
    final_optimized["priority_category"] == "High"
).sum()

print(f"Critical tasks scheduled   : {critical_count:,}")
print(f"High-priority scheduled    : {high_count:,}")

# Train conflicts
total_conflicts = int(
    final_optimized["train_conflict_count"].sum()
)

print(f"Train conflicts            : {total_conflicts:,}")

# Conflict duration
if "conflict_duration_min" in final_optimized.columns:

    total_conflict_duration = (
        final_optimized["conflict_duration_min"].sum()
    )

elif "total_conflict_duration_min" in final_optimized.columns:

    total_conflict_duration = (
        final_optimized["total_conflict_duration_min"].sum()
    )

else:

    total_conflict_duration = 0

print(
    f"Conflict duration          : "
    f"{total_conflict_duration:.0f} minutes"
)

# Solver status if available
if "solver_status" in globals():
    print(f"Solver status              : {solver_status}")

print("=" * 70)
print("Optimization stage completed successfully.")
print("=" * 70)

RAILWISE — OPTIMIZATION SUMMARY
Maintenance tasks          : 14,400
Block requests             : 6,000
Available COA windows      : 1,366
Candidate blocks           : 1,204
Tasks with candidates      : 749
Optimized blocks           : 258
Critical tasks scheduled   : 6
High-priority scheduled    : 47
Train conflicts            : 5
Conflict duration          : 38 minutes
Optimization stage completed successfully.


In [27]:
coa_lookup = available_coa[
    [
        "block_window_id",
        "available_duration_min"
    ]
].drop_duplicates("block_window_id")

evaluation = final_optimized.merge(
    coa_lookup,
    on="block_window_id",
    how="left"
)

evaluation["optimized_utilization_pct"] = (
    evaluation["recommended_duration_min"]
    / evaluation["available_duration_min"]
    * 100
)

evaluation["optimized_utilization_pct"] = (
    evaluation["optimized_utilization_pct"]
    .clip(upper=100)
    .round(2)
)

print(
    "Average optimized utilization:",
    round(
        evaluation["optimized_utilization_pct"].mean(),
        2
    ),
    "%"
)

Average optimized utilization: 68.14 %


In [28]:
# ============================================================
# RECOVER EVALUATION DATAFRAME
# ============================================================

# Rebuild evaluation directly from optimized_blocks
# This avoids stale final_optimized/evaluation objects.

evaluation = optimized_blocks.copy()

# Make sure block_request_id exists
required_cols = [
    "block_request_id",
    "task_id",
    "asset_id",
    "department",
    "corridor_id",
    "block_window_id",
    "block_date",
    "predicted_priority",
    "priority_category",
    "recommended_start",
    "recommended_end",
    "duration_min",
    "train_conflicts",
    "total_conflict_duration_min"
]

missing = [
    col for col in required_cols
    if col not in evaluation.columns
]

print("Missing columns:", missing)

if missing:
    raise ValueError(
        f"optimized_blocks is missing: {missing}. "
        "Do not rerun the optimizer; check Cell 20 output."
    )

# ------------------------------------------------------------
# Add COA capacity
# ------------------------------------------------------------

coa_lookup = available_coa[
    [
        "block_window_id",
        "available_duration_min"
    ]
].drop_duplicates("block_window_id")

evaluation = evaluation.merge(
    coa_lookup,
    on="block_window_id",
    how="left"
)

# ------------------------------------------------------------
# Calculate real utilization
# ------------------------------------------------------------

evaluation["optimized_utilization_pct"] = (
    evaluation["duration_min"]
    / evaluation["available_duration_min"]
    * 100
)

evaluation["optimized_utilization_pct"] = (
    evaluation["optimized_utilization_pct"]
    .clip(upper=100)
    .round(2)
)

print("Evaluation shape:", evaluation.shape)
print("Evaluation columns:")
print(evaluation.columns.tolist())

display(
    evaluation[
        [
            "block_request_id",
            "task_id",
            "corridor_id",
            "block_window_id",
            "predicted_priority",
            "train_conflicts",
            "total_conflict_duration_min",
            "optimized_utilization_pct"
        ]
    ].head(10)
)

Missing columns: []
Evaluation shape: (258, 24)
Evaluation columns:
['block_request_id', 'task_id', 'asset_id', 'corridor_id', 'block_window_id', 'block_date', 'start_min', 'end_min', 'duration_min', 'predicted_priority', 'priority_category', 'department', 'possession_required', 'candidate_id', 'train_conflicts', 'total_conflict_duration_min', 'forecast_date', 'forecast_goods_trains', 'peak_period', 'confidence_pct', 'recommended_start', 'recommended_end', 'available_duration_min', 'optimized_utilization_pct']


,block_request_id,task_id,corridor_id,block_window_id,predicted_priority,train_conflicts,total_conflict_duration_min,optimized_utilization_pct
0,BR000004,TMS000005,C00078,COA000722,67.689354,0,0.0,69.64
1,BR000025,TMS000028,C00076,COA001405,59.352386,0,0.0,64.71
2,BR000044,TMS000055,C00066,COA000597,26.743483,0,0.0,23.24
3,BR000051,TMS000065,C00001,COA001199,77.893135,0,0.0,76.83
4,BR000062,TMS000076,C00046,COA000796,54.495583,0,0.0,79.87
5,BR000070,TMS000086,C00017,COA000826,74.352638,0,0.0,68.83
6,BR000081,TMS000098,C00006,COA001192,37.731422,0,0.0,41.46
7,BR000086,TMS000103,C00050,COA001121,30.329281,0,0.0,28.66
8,BR000110,TMS000132,C00008,COA001474,27.067982,0,0.0,51.05
9,BR000115,TMS000141,C00093,COA001291,44.866592,0,0.0,78.72


In [29]:
# ============================================================
# HISTORICAL BASELINE COMPARISON
# ============================================================

historical_eval = historical.copy()

historical_eval.columns = (
    historical_eval.columns
    .astype(str)
    .str.strip()
)

historical_eval["planned_date"] = pd.to_datetime(
    historical_eval["planned_date"]
)

# ------------------------------------------------------------
# Aggregate historical plans per block request + corridor
# ------------------------------------------------------------

historical_summary = (
    historical_eval
    .groupby(
        ["block_request_id", "corridor_id"],
        as_index=False
    )
    .agg(
        historical_conflict_count=(
            "conflict_count",
            "mean"
        ),
        historical_utilization_pct=(
            "utilization_pct",
            "mean"
        ),
        historical_duration_min=(
            "planned_duration_min",
            "mean"
        ),
        historical_planning_method=(
            "planning_method",
            "first"
        ),
        historical_status=(
            "status",
            "first"
        )
    )
)

print(
    "Unique historical request/corridor combinations:",
    len(historical_summary)
)

# ------------------------------------------------------------
# Merge with optimized result
# ------------------------------------------------------------

comparison = evaluation.merge(
    historical_summary,
    on=[
        "block_request_id",
        "corridor_id"
    ],
    how="left"
)

print("Optimized records:", len(evaluation))

print(
    "Historical matches:",
    comparison["historical_planning_method"].notna().sum()
)

display(
    comparison[
        [
            "block_request_id",
            "task_id",
            "corridor_id",
            "predicted_priority",
            "train_conflicts",
            "historical_conflict_count",
            "optimized_utilization_pct",
            "historical_utilization_pct",
            "historical_planning_method",
            "historical_status"
        ]
    ].head(20)
)

Unique historical request/corridor combinations: 4000
Optimized records: 258
Historical matches: 178


,block_request_id,task_id,corridor_id,predicted_priority,train_conflicts,historical_conflict_count,optimized_utilization_pct,historical_utilization_pct,historical_planning_method,historical_status
0,BR000004,TMS000005,C00078,67.689354,0,NaN,69.64,NaN,NaN,NaN
1,BR000025,TMS000028,C00076,59.352386,0,1.0,64.71,50.0,Manual,Completed
2,BR000044,TMS000055,C00066,26.743483,0,2.0,23.24,80.0,Manual,Completed
3,BR000051,TMS000065,C00001,77.893135,0,3.0,76.83,69.0,Manual,Completed
4,BR000062,TMS000076,C00046,54.495583,0,2.0,79.87,70.0,Manual,Completed
5,BR000070,TMS000086,C00017,74.352638,0,1.0,68.83,85.0,Manual,Completed
6,BR000081,TMS000098,C00006,37.731422,0,4.0,41.46,67.0,Manual,Completed
7,BR000086,TMS000103,C00050,30.329281,0,NaN,28.66,NaN,NaN,NaN
8,BR000110,TMS000132,C00008,27.067982,0,1.0,51.05,48.0,Manual,Completed
9,BR000115,TMS000141,C00093,44.866592,0,NaN,78.72,NaN,NaN,NaN


In [30]:
# ============================================================
# BASELINE VS OPTIMIZED METRICS
# ============================================================

matched = comparison[
    comparison["historical_planning_method"].notna()
].copy()

print("=" * 65)
print("RAILWISE — HISTORICAL VS OPTIMIZED")
print("=" * 65)

print("Optimized blocks:", len(evaluation))
print("Historical matches:", len(matched))

if len(matched) > 0:

    historical_conflicts = (
        matched["historical_conflict_count"].sum()
    )

    optimized_conflicts = (
        matched["train_conflicts"].sum()
    )

    historical_utilization = (
        matched["historical_utilization_pct"].mean()
    )

    optimized_utilization = (
        matched["optimized_utilization_pct"].mean()
    )

    print()
    print(
        "Historical total conflicts:",
        round(historical_conflicts, 2)
    )

    print(
        "Optimized total conflicts:",
        optimized_conflicts
    )

    print(
        "Historical average utilization:",
        round(historical_utilization, 2),
        "%"
    )

    print(
        "Optimized average utilization:",
        round(optimized_utilization, 2),
        "%"
    )

    if historical_conflicts > 0:

        conflict_reduction = (
            (historical_conflicts - optimized_conflicts)
            / historical_conflicts
            * 100
        )

        print(
            "Conflict reduction:",
            round(conflict_reduction, 2),
            "%"
        )

    utilization_change = (
        optimized_utilization
        - historical_utilization
    )

    print(
        "Utilization change:",
        round(utilization_change, 2),
        "percentage points"
    )

else:

    print()
    print("No historical matches found.")

RAILWISE — HISTORICAL VS OPTIMIZED
Optimized blocks: 258
Historical matches: 178

Historical total conflicts: 326.0
Optimized total conflicts: 5
Historical average utilization: 65.31 %
Optimized average utilization: 68.49 %
Conflict reduction: 98.47 %
Utilization change: 3.18 percentage points


In [31]:
# ============================================================
# CELL 30 — SAVE BASELINE COMPARISON
# ============================================================

baseline_comparison = matched.copy()

if len(baseline_comparison) > 0:

    baseline_comparison["conflict_change"] = (
        baseline_comparison["train_conflicts"]
        - baseline_comparison["historical_conflict_count"]
    )

    baseline_comparison[
        "utilization_change_pct_points"
    ] = (
        baseline_comparison[
            "optimized_utilization_pct"
        ]
        -
        baseline_comparison[
            "historical_utilization_pct"
        ]
    )

baseline_path = os.path.join(
    OUTPUT_DIR,
    "baseline_comparison.csv"
)

baseline_comparison.to_csv(
    baseline_path,
    index=False
)

print("Saved:", baseline_path)

Saved: C:\Users\diksh\OneDrive\Desktop\RailWise\outputs\baseline_comparison.csv


In [33]:
# ============================================================
# CELL 31 — FINAL RAILWISE OPTIMIZATION SUMMARY
# ============================================================

print("=" * 70)
print("RAILWISE — FINAL OPTIMIZATION SUMMARY")
print("=" * 70)

print(f"Maintenance tasks          : {len(maintenance):,}")
print(f"Block requests             : {len(block_requests):,}")
print(f"Available COA windows      : {len(available_coa):,}")
print(f"Candidate blocks           : {len(opt_candidates):,}")
print(f"Tasks with candidates      : {opt_candidates['task_id'].nunique():,}")
print(f"Optimized blocks           : {len(final_optimized):,}")

critical_count = (
    final_optimized["priority_category"] == "Critical"
).sum()

high_count = (
    final_optimized["priority_category"] == "High"
).sum()

print(f"Critical tasks scheduled   : {critical_count:,}")
print(f"High-priority scheduled    : {high_count:,}")

# ------------------------------------------------------------
# Conflict metrics
# ------------------------------------------------------------

total_conflicts = int(
    final_optimized["train_conflict_count"].sum()
)

# Handle either possible column name
if "conflict_duration_min" in final_optimized.columns:

    total_conflict_duration = (
        final_optimized["conflict_duration_min"].sum()
    )

elif "total_conflict_duration_min" in final_optimized.columns:

    total_conflict_duration = (
        final_optimized["total_conflict_duration_min"].sum()
    )

else:

    total_conflict_duration = 0

print(f"Train conflicts            : {total_conflicts:,}")
print(
    f"Conflict duration          : "
    f"{total_conflict_duration:.0f} minutes"
)

# ------------------------------------------------------------
# Utilization
# ------------------------------------------------------------

if "optimized_utilization_pct" in evaluation.columns:

    avg_utilization = (
        evaluation["optimized_utilization_pct"].mean()
    )

    print(
        f"Avg block utilization      : "
        f"{avg_utilization:.2f}%"
    )

# ============================================================
# HISTORICAL COMPARISON
# ============================================================

print()
print("--- Historical Comparison ---")

if len(matched) > 0:

    # Calculate metrics
    historical_conflicts = (
        matched["historical_conflict_count"].sum()
    )

    optimized_conflicts = (
        matched["train_conflicts"].sum()
    )

    historical_utilization = (
        matched["historical_utilization_pct"].mean()
    )

    optimized_utilization = (
        matched["optimized_utilization_pct"].mean()
    )

    # Conflict reduction
    if historical_conflicts > 0:

        conflict_reduction_pct = (
            (historical_conflicts - optimized_conflicts)
            / historical_conflicts
            * 100
        )

    else:

        conflict_reduction_pct = np.nan

    # Utilization change
    utilization_change = (
        optimized_utilization
        - historical_utilization
    )

    print(
        f"Historical conflicts       : "
        f"{historical_conflicts:.2f}"
    )

    print(
        f"Optimized conflicts        : "
        f"{optimized_conflicts:.0f}"
    )

    if not np.isnan(conflict_reduction_pct):

        print(
            f"Conflict reduction         : "
            f"{conflict_reduction_pct:.2f}%"
        )

    print(
        f"Historical utilization     : "
        f"{historical_utilization:.2f}%"
    )

    print(
        f"Optimized utilization      : "
        f"{optimized_utilization:.2f}%"
    )

    print(
        f"Utilization change         : "
        f"{utilization_change:.2f} percentage points"
    )

else:

    print("No historical matches found.")

RAILWISE — FINAL OPTIMIZATION SUMMARY
Maintenance tasks          : 14,400
Block requests             : 6,000
Available COA windows      : 1,366
Candidate blocks           : 1,204
Tasks with candidates      : 749
Optimized blocks           : 258
Critical tasks scheduled   : 6
High-priority scheduled    : 47
Train conflicts            : 5
Conflict duration          : 38 minutes
Avg block utilization      : 68.14%

--- Historical Comparison ---
Historical conflicts       : 326.00
Optimized conflicts        : 5
Conflict reduction         : 98.47%
Historical utilization     : 65.31%
Optimized utilization      : 68.49%
Utilization change         : 3.18 percentage points


In [34]:
# ============================================================
# RAILWISE — REQUEST STATUS CLASSIFICATION
# ============================================================

all_requests = block_requests.copy()

candidate_request_ids = (
    opt_candidates["block_request_id"]
    .dropna()
    .unique()
)

scheduled_request_ids = (
    final_optimized["block_request_id"]
    .dropna()
    .unique()
)

selected_requests = all_requests[
    all_requests["block_request_id"].isin(scheduled_request_ids)
].copy()

deferred_requests = all_requests[
    all_requests["block_request_id"].isin(candidate_request_ids)
    & ~all_requests["block_request_id"].isin(scheduled_request_ids)
].copy()

no_candidate_requests = all_requests[
    ~all_requests["block_request_id"].isin(candidate_request_ids)
].copy()

print("=" * 70)
print("RAILWISE — REQUEST STATUS")
print("=" * 70)
print(f"Total requests                 : {len(all_requests):,}")
print(f"Requests with candidates       : {len(candidate_request_ids):,}")
print(f"Selected / optimized           : {len(selected_requests):,}")
print(f"Deferred / not selected        : {len(deferred_requests):,}")
print(f"No feasible candidate          : {len(no_candidate_requests):,}")
print(
    f"Coverage check                 : "
    f"{len(selected_requests) + len(deferred_requests) + len(no_candidate_requests):,}"
)
print("=" * 70)


RAILWISE — REQUEST STATUS
Total requests                 : 6,000
Requests with candidates       : 749
Selected / optimized           : 258
Deferred / not selected        : 491
No feasible candidate          : 5,251
Coverage check                 : 6,000


In [98]:
# ============================================================
# RAILWISE — BUILD FINAL RECOMMENDATIONS
# ============================================================

# ------------------------------------------------------------
# 1. CREATE BLOCK REQUEST MAPPING
# ------------------------------------------------------------

request_mapping = (
    opt_candidates[
        [
            "block_request_id",
            "task_id",
            "block_window_id"
        ]
    ]
    .drop_duplicates(
        [
            "block_request_id",
            "task_id",
            "block_window_id"
        ]
    )
)

# ------------------------------------------------------------
# 2. OPTIMIZED REQUESTS
# ------------------------------------------------------------

optimized_final = final_optimized.copy()

# Add block_request_id BEFORE using it
optimized_final = optimized_final.merge(
    request_mapping,
    on=["task_id", "block_window_id"],
    how="left"
)

# Check mapping
if optimized_final["block_request_id"].isna().any():
    print("WARNING: Some optimized blocks could not be mapped.")

# Exactly one row per block request
optimized_final = (
    optimized_final
    .drop_duplicates("block_request_id")
    .copy()
)

optimized_final["request_status"] = "OPTIMIZED"

# ------------------------------------------------------------
# 3. CONFLICT DURATION
# ------------------------------------------------------------

conflict_lookup = (
    opt_candidates[
        [
            "block_request_id",
            "task_id",
            "block_window_id",
            "total_conflict_duration_min"
        ]
    ]
    .drop_duplicates(
        [
            "block_request_id",
            "task_id",
            "block_window_id"
        ]
    )
)

optimized_final = optimized_final.merge(
    conflict_lookup,
    on=[
        "block_request_id",
        "task_id",
        "block_window_id"
    ],
    how="left"
)

optimized_final["conflict_duration_min"] = (
    optimized_final["total_conflict_duration_min"]
    .fillna(0)
)

# ------------------------------------------------------------
# 4. AFFECTED TRAINS
# ------------------------------------------------------------

affected_trains = (
    conflict_analysis[
        [
            "candidate_id",
            "train_no"
        ]
    ]
    .drop_duplicates()
    .groupby("candidate_id")["train_no"]
    .apply(
        lambda x: ", ".join(
            sorted(
                x.astype(str).unique()
            )
        )
    )
    .reset_index(
        name="affected_trains"
    )
)

# Get candidate_id for optimized blocks
candidate_mapping = (
    opt_candidates[
        [
            "block_request_id",
            "task_id",
            "block_window_id",
            "candidate_id"
        ]
    ]
    .drop_duplicates(
        [
            "block_request_id",
            "task_id",
            "block_window_id"
        ]
    )
)

optimized_final = optimized_final.merge(
    candidate_mapping,
    on=[
        "block_request_id",
        "task_id",
        "block_window_id"
    ],
    how="left"
)

optimized_final = optimized_final.merge(
    affected_trains,
    on="candidate_id",
    how="left"
)

optimized_final["affected_trains"] = (
    optimized_final["affected_trains"]
    .fillna("")
)

# ------------------------------------------------------------
# 5. DEFERRED REQUESTS
# ------------------------------------------------------------

deferred_final = (
    deferred_requests
    .drop_duplicates("block_request_id")
    .copy()
)

deferred_final = deferred_final.merge(
    maintenance_lookup,
    on=["task_id", "corridor_id"],
    how="left"
)

deferred_final["predicted_priority"] = (
    deferred_final["maintenance_priority_score"]
)

deferred_final["priority_category"] = (
    deferred_final["predicted_priority"]
    .apply(get_priority_category)
)

deferred_final["recommended_date"] = pd.NaT
deferred_final["recommended_start"] = None
deferred_final["recommended_end"] = None

deferred_final["recommended_duration_min"] = (
    deferred_final["requested_duration_min"]
)

deferred_final["train_conflict_count"] = 0
deferred_final["conflict_duration_min"] = 0
deferred_final["forecast_goods_trains"] = 0
deferred_final["affected_trains"] = ""

deferred_final["feasibility"] = "DEFERRED"

deferred_final["request_status"] = "DEFERRED"

deferred_final["reason_for_recommendation"] = (
    "Candidate block windows were available, but the request "
    "was not selected by the global optimization solution."
)

# ------------------------------------------------------------
# 6. NO FEASIBLE REQUESTS
# ------------------------------------------------------------

no_feasible_final = (
    no_candidate_requests
    .drop_duplicates("block_request_id")
    .copy()
)

no_feasible_final = no_feasible_final.merge(
    maintenance_lookup,
    on=["task_id", "corridor_id"],
    how="left"
)

no_feasible_final["predicted_priority"] = (
    no_feasible_final["maintenance_priority_score"]
)

no_feasible_final["priority_category"] = (
    no_feasible_final["predicted_priority"]
    .apply(get_priority_category)
)

no_feasible_final["recommended_date"] = pd.NaT
no_feasible_final["recommended_start"] = None
no_feasible_final["recommended_end"] = None

no_feasible_final["recommended_duration_min"] = (
    no_feasible_final["requested_duration_min"]
)

no_feasible_final["train_conflict_count"] = 0
no_feasible_final["conflict_duration_min"] = 0
no_feasible_final["forecast_goods_trains"] = 0
no_feasible_final["affected_trains"] = ""

no_feasible_final["feasibility"] = (
    "NO FEASIBLE BLOCK"
)

no_feasible_final["request_status"] = (
    "NO FEASIBLE BLOCK"
)

no_feasible_final["reason_for_recommendation"] = (
    "No candidate block window satisfying the current "
    "candidate-generation conditions was available."
)

# ------------------------------------------------------------
# 7. STANDARD OUTPUT COLUMNS
# ------------------------------------------------------------

final_columns = [
    "block_request_id",
    "task_id",
    "asset_id",
    "department",
    "corridor_id",
    "predicted_priority",
    "priority_category",
    "requested_duration_min",
    "recommended_date",
    "recommended_start",
    "recommended_end",
    "recommended_duration_min",
    "train_conflict_count",
    "conflict_duration_min",
    "forecast_goods_trains",
    "affected_trains",
    "feasibility",
    "request_status",
    "reason_for_recommendation"
]

def standardize(df):

    for col in final_columns:
        if col not in df.columns:
            df[col] = np.nan

    return df[final_columns].copy()

optimized_final = standardize(optimized_final)
deferred_final = standardize(deferred_final)
no_feasible_final = standardize(no_feasible_final)

# ------------------------------------------------------------
# 8. COMBINE
# ------------------------------------------------------------

railwise_final = pd.concat(
    [
        optimized_final,
        deferred_final,
        no_feasible_final
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# 9. FINAL DUPLICATE PROTECTION
# ------------------------------------------------------------

railwise_final = (
    railwise_final
    .drop_duplicates("block_request_id")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 10. VALIDATION
# ------------------------------------------------------------

print("=" * 70)
print("RAILWISE — FINAL RECOMMENDATION DATASET")
print("=" * 70)

print(
    f"Total records      : "
    f"{len(railwise_final):,}"
)

print(
    f"Unique requests    : "
    f"{railwise_final['block_request_id'].nunique():,}"
)

print(
    f"Duplicate requests : "
    f"{railwise_final['block_request_id'].duplicated().sum():,}"
)

print("\nRequest status:")

print(
    railwise_final[
        "request_status"
    ].value_counts()
)

print("=" * 70)

RAILWISE — FINAL RECOMMENDATION DATASET
Total records      : 6,000
Unique requests    : 6,000
Duplicate requests : 0

Request status:
request_status
NO FEASIBLE BLOCK    5251
DEFERRED              491
OPTIMIZED             258
Name: count, dtype: int64


In [36]:
# ============================================================
# RAILWISE — OPERATIONAL IMPACT SCORE
# Calculated only for actual optimized blocks
# ============================================================

impact_df = railwise_final.copy()

corridor_lookup = (
    corridors[
        ["corridor_id", "corridor_criticality"]
    ]
    .drop_duplicates("corridor_id")
)

impact_df = impact_df.merge(
    corridor_lookup,
    on="corridor_id",
    how="left"
)

impact_df["corridor_criticality"] = pd.to_numeric(
    impact_df["corridor_criticality"],
    errors="coerce"
)

selected_mask = (
    impact_df["request_status"] == "OPTIMIZED"
)

def minmax_selected(series):
    result = pd.Series(
        np.nan,
        index=series.index,
        dtype=float
    )

    values = pd.to_numeric(
        series[selected_mask],
        errors="coerce"
    ).fillna(0)

    if len(values) == 0:
        return result

    if values.max() == values.min():
        result.loc[selected_mask] = 0.0
    else:
        result.loc[selected_mask] = (
            (values - values.min())
            / (values.max() - values.min())
            * 100
        )

    return result

priority_component = minmax_selected(
    impact_df["predicted_priority"]
)

conflict_component = minmax_selected(
    impact_df["train_conflict_count"]
)

duration_component = minmax_selected(
    impact_df["conflict_duration_min"]
)

goods_component = minmax_selected(
    impact_df["forecast_goods_trains"]
)

corridor_component = minmax_selected(
    impact_df["corridor_criticality"]
)

impact_df["operational_impact_score"] = (
    0.30 * priority_component
    + 0.25 * conflict_component
    + 0.20 * duration_component
    + 0.15 * goods_component
    + 0.10 * corridor_component
).round(2)

def impact_category(score):
    if pd.isna(score):
        return "N/A"
    if score >= 75:
        return "Very High"
    if score >= 50:
        return "High"
    if score >= 25:
        return "Medium"
    return "Low"

impact_df["operational_impact_category"] = (
    impact_df["operational_impact_score"]
    .apply(impact_category)
)

railwise_final = impact_df.copy()

print("=" * 70)
print("RAILWISE — OPERATIONAL IMPACT")
print("=" * 70)
print(
    f"Optimized blocks scored : "
    f"{selected_mask.sum():,}"
)
print(
    f"Average optimized impact: "
    f"{railwise_final.loc[selected_mask, 'operational_impact_score'].mean():.2f}"
)
print("\nImpact categories:")
print(
    railwise_final.loc[
        selected_mask,
        "operational_impact_category"
    ].value_counts()
)
print("=" * 70)


RAILWISE — OPERATIONAL IMPACT
Optimized blocks scored : 361
Average optimized impact: 23.18

Impact categories:
operational_impact_category
Low          215
Medium       141
High           3
Very High      2
Name: count, dtype: int64


In [38]:
# ============================================================
# RAILWISE — FINAL TECHNICAL INTEGRATION CHECK
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("../")

if not (PROJECT_ROOT / "outputs").exists():
    PROJECT_ROOT = Path(".")

OUTPUT_DIR = PROJECT_ROOT / "outputs"

files_to_check = {
    "asset_risk_predictions.csv": 14400,
    "candidate_blocks.csv": 1204,
    "conflict_analysis.csv": 743,
    "optimized_blocks.csv": 258,
    "baseline_comparison.csv": 178,
    "railwise_final_recommendations.csv": 6000
}

print("=" * 75)
print("RAILWISE — FINAL TECHNICAL INTEGRATION CHECK")
print("=" * 75)

all_ok = True

for filename, expected_rows in files_to_check.items():

    path = OUTPUT_DIR / filename

    if not path.exists():
        print(f"✗ {filename} — FILE NOT FOUND")
        all_ok = False
        continue

    df = pd.read_csv(path)

    row_ok = len(df) == expected_rows

    status = "✓" if row_ok else "⚠"

    print(
        f"{status} {filename:<40} "
        f"{len(df):>6,} rows"
    )

    if not row_ok:
        print(
            f"    Expected: {expected_rows:,} rows"
        )
        all_ok = False

print("=" * 75)

if all_ok:
    print("✓ ALL PIPELINE OUTPUTS PASSED")
    print("✓ READY FOR APPLICATION INTEGRATION")
else:
    print("⚠ OUTPUT VALIDATION NEEDS ATTENTION")

print("=" * 75)

RAILWISE — FINAL TECHNICAL INTEGRATION CHECK
✓ asset_risk_predictions.csv               14,400 rows
✓ candidate_blocks.csv                      1,204 rows
✓ conflict_analysis.csv                       743 rows
✓ optimized_blocks.csv                        258 rows
✓ baseline_comparison.csv                     178 rows
⚠ railwise_final_recommendations.csv        6,103 rows
    Expected: 6,000 rows
⚠ OUTPUT VALIDATION NEEDS ATTENTION


In [100]:
# ============================================================
# RAILWISE — FINAL DATA QUALITY CHECK
# ============================================================

print("=" * 70)
print("RAILWISE — FINAL DATA QUALITY CHECK")
print("=" * 70)

print("Rows:", len(railwise_final))
print(
    "Unique block requests:",
    railwise_final["block_request_id"].nunique()
)

print(
    "Duplicate block requests:",
    railwise_final["block_request_id"].duplicated().sum()
)

print(
    "Missing block_request_id:",
    railwise_final["block_request_id"].isna().sum()
)

print("\nStatus counts:")
print(
    railwise_final["request_status"].value_counts()
)

print("\nOptimized records:")
display(
    railwise_final[
        railwise_final["request_status"] == "OPTIMIZED"
    ][
        [
            "block_request_id",
            "task_id",
            "asset_id",
            "corridor_id",
            "predicted_priority",
            "priority_category",
            "recommended_date",
            "recommended_start",
            "recommended_end",
            "recommended_duration_min",
            "train_conflict_count",
            "conflict_duration_min",
            "feasibility"
        ]
    ].head(10)
)

print("=" * 70)

RAILWISE — FINAL DATA QUALITY CHECK
Rows: 6000
Unique block requests: 6000
Duplicate block requests: 0
Missing block_request_id: 0

Status counts:
request_status
NO FEASIBLE BLOCK    5251
DEFERRED              491
OPTIMIZED             258
Name: count, dtype: int64

Optimized records:


,block_request_id,task_id,asset_id,corridor_id,predicted_priority,priority_category,recommended_date,recommended_start,recommended_end,recommended_duration_min,train_conflict_count,conflict_duration_min,feasibility
0,BR000004,TMS000005,AST001012,C00078,67.689354,High,2025-06-04,16:28,17:46,78,0,0.0,Feasible - No Train Conflict
1,BR000025,TMS000028,AST001133,C00076,59.352386,Medium,2025-06-02,19:06,20:56,110,0,0.0,Feasible - No Train Conflict
2,BR000044,TMS000055,AST004019,C00066,26.743483,Low,2025-06-25,01:08,01:41,33,0,0.0,Feasible - No Train Conflict
3,BR000051,TMS000065,AST000013,C00001,77.893135,High,2025-06-02,13:23,14:26,63,0,0.0,Feasible - No Train Conflict
4,BR000062,TMS000076,AST002317,C00046,54.495583,Medium,2025-06-15,05:59,07:58,119,0,0.0,Feasible - No Train Conflict
5,BR000070,TMS000086,AST000036,C00017,74.352638,High,2025-06-20,03:06,03:59,53,0,0.0,Feasible - No Train Conflict
6,BR000081,TMS000098,AST003042,C00006,37.731422,Low,2025-06-01,00:40,01:14,34,0,0.0,Feasible - No Train Conflict
7,BR000086,TMS000103,AST004028,C00050,30.329281,Low,2025-06-19,17:23,18:10,47,0,0.0,Feasible - No Train Conflict
8,BR000110,TMS000132,AST003824,C00008,27.067982,Low,2025-06-10,19:01,20:14,73,0,0.0,Feasible - No Train Conflict
9,BR000115,TMS000141,AST001048,C00093,44.866592,Medium,2025-06-13,12:48,14:02,74,0,0.0,Feasible - No Train Conflict


In [101]:
# ============================================================
# RAILWISE — SAVE CLEAN FINAL RECOMMENDATIONS
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# 1. DEFINE OUTPUT DIRECTORY
# ------------------------------------------------------------

OUTPUT_DIR = Path("../outputs")

# If notebook is being run from a different working directory
if not OUTPUT_DIR.exists():
    OUTPUT_DIR = Path("outputs")

# Create directory if it does not exist
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. DEFINE FINAL OUTPUT PATH
# ------------------------------------------------------------

final_output_path = (
    OUTPUT_DIR /
    "railwise_final_recommendations.csv"
)

# ------------------------------------------------------------
# 3. SAFETY CHECKS BEFORE SAVING
# ------------------------------------------------------------

assert "railwise_final" in globals(), (
    "ERROR: railwise_final does not exist. "
    "Run the FINAL DATA QUALITY / FINAL RECOMMENDATIONS cell first."
)

assert len(railwise_final) == 6000, (
    f"ERROR: Expected 6000 rows, "
    f"but found {len(railwise_final)}."
)

assert (
    railwise_final["block_request_id"].nunique() == 6000
), (
    "ERROR: block_request_id values are not unique."
)

assert (
    railwise_final["block_request_id"].isna().sum() == 0
), (
    "ERROR: Missing block_request_id values found."
)

# ------------------------------------------------------------
# 4. SAVE CSV
# ------------------------------------------------------------

railwise_final.to_csv(
    final_output_path,
    index=False
)

# ------------------------------------------------------------
# 5. VERIFY FILE
# ------------------------------------------------------------

print("=" * 70)
print("RAILWISE — FINAL RECOMMENDATIONS SAVED")
print("=" * 70)

print(
    f"File path : {final_output_path.resolve()}"
)

print(
    f"Rows      : {len(railwise_final):,}"
)

print(
    f"Columns   : {len(railwise_final.columns):,}"
)

print(
    f"Unique IDs: "
    f"{railwise_final['block_request_id'].nunique():,}"
)

print("\nRequest status:")

print(
    railwise_final[
        "request_status"
    ].value_counts()
)

print("\nFile exists:")

print(
    final_output_path.exists()
)

print("=" * 70)

RAILWISE — FINAL RECOMMENDATIONS SAVED
File path : C:\Users\diksh\OneDrive\Desktop\RailWise\outputs\railwise_final_recommendations.csv
Rows      : 6,000
Columns   : 19
Unique IDs: 6,000

Request status:
request_status
NO FEASIBLE BLOCK    5251
DEFERRED              491
OPTIMIZED             258
Name: count, dtype: int64

File exists:
True


In [102]:
# ============================================================
# RAILWISE — FINAL BACKEND VALIDATION
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. OUTPUT DIRECTORY
# ------------------------------------------------------------

OUTPUT_DIR = Path("../outputs")

if not OUTPUT_DIR.exists():
    OUTPUT_DIR = Path("outputs")

print("=" * 80)
print("RAILWISE — FINAL BACKEND VALIDATION")
print("=" * 80)

# ------------------------------------------------------------
# 2. REQUIRED OUTPUT FILES
# ------------------------------------------------------------

required_files = {
    "Asset Risk Predictions": "asset_risk_predictions.csv",
    "Candidate Blocks": "candidate_blocks.csv",
    "Conflict Analysis": "conflict_analysis.csv",
    "Optimized Blocks": "optimized_blocks.csv",
    "Baseline Comparison": "baseline_comparison.csv",
    "Final Recommendations": "railwise_final_recommendations.csv"
}

validation_results = []

for name, filename in required_files.items():

    path = OUTPUT_DIR / filename

    exists = path.exists()

    if exists:
        df = pd.read_csv(path)

        validation_results.append({
            "Output": name,
            "File": filename,
            "Exists": "YES",
            "Rows": len(df),
            "Columns": len(df.columns)
        })

        print(
            f"[OK] {filename:<40} "
            f"Rows: {len(df):>6,} | "
            f"Columns: {len(df.columns):>3}"
        )

    else:

        validation_results.append({
            "Output": name,
            "File": filename,
            "Exists": "NO",
            "Rows": 0,
            "Columns": 0
        })

        print(
            f"[MISSING] {filename}"
        )

# ------------------------------------------------------------
# 3. FINAL RECOMMENDATION VALIDATION
# ------------------------------------------------------------

final_path = OUTPUT_DIR / "railwise_final_recommendations.csv"

if final_path.exists():

    final_df = pd.read_csv(final_path)

    print("\n" + "-" * 80)
    print("FINAL RECOMMENDATION CHECK")
    print("-" * 80)

    print(
        f"Rows                 : {len(final_df):,}"
    )

    print(
        f"Unique block requests: "
        f"{final_df['block_request_id'].nunique():,}"
    )

    print(
        f"Duplicate IDs        : "
        f"{final_df['block_request_id'].duplicated().sum():,}"
    )

    print(
        f"Missing IDs          : "
        f"{final_df['block_request_id'].isna().sum():,}"
    )

    print("\nRequest status:")

    print(
        final_df["request_status"].value_counts()
    )

# ------------------------------------------------------------
# 4. OPTIMIZED BLOCK VALIDATION
# ------------------------------------------------------------

optimized_path = OUTPUT_DIR / "optimized_blocks.csv"

if optimized_path.exists():

    optimized_df = pd.read_csv(
        optimized_path
    )

    print("\n" + "-" * 80)
    print("OPTIMIZED BLOCK CHECK")
    print("-" * 80)

    print(
        f"Optimized blocks : {len(optimized_df):,}"
    )

    if "train_conflict_count" in optimized_df.columns:

        print(
            f"Train conflicts : "
            f"{optimized_df['train_conflict_count'].sum():,.0f}"
        )

    elif "train_conflicts" in optimized_df.columns:

        print(
            f"Train conflicts : "
            f"{optimized_df['train_conflicts'].sum():,.0f}"
        )

    if "conflict_duration_min" in optimized_df.columns:

        print(
            f"Conflict duration: "
            f"{optimized_df['conflict_duration_min'].sum():,.0f} min"
        )

    elif "total_conflict_duration_min" in optimized_df.columns:

        print(
            f"Conflict duration: "
            f"{optimized_df['total_conflict_duration_min'].sum():,.0f} min"
        )

# ------------------------------------------------------------
# 5. BASELINE VALIDATION
# ------------------------------------------------------------

baseline_path = OUTPUT_DIR / "baseline_comparison.csv"

if baseline_path.exists():

    baseline_df = pd.read_csv(
        baseline_path
    )

    print("\n" + "-" * 80)
    print("HISTORICAL BASELINE CHECK")
    print("-" * 80)

    print(
        f"Baseline records : {len(baseline_df):,}"
    )

    print(
        "Columns:"
    )

    print(
        list(baseline_df.columns)
    )

# ------------------------------------------------------------
# 6. FINAL SUMMARY
# ------------------------------------------------------------

all_exist = all(
    (OUTPUT_DIR / filename).exists()
    for filename in required_files.values()
)

print("\n" + "=" * 80)

if all_exist:
    print("STATUS: ALL REQUIRED OUTPUT FILES ARE READY")
else:
    print("STATUS: SOME OUTPUT FILES ARE MISSING")

print("=" * 80)

# ------------------------------------------------------------
# 7. VALIDATION TABLE
# ------------------------------------------------------------

validation_df = pd.DataFrame(
    validation_results
)

display(validation_df)

RAILWISE — FINAL BACKEND VALIDATION
[OK] asset_risk_predictions.csv               Rows: 14,400 | Columns:  13
[OK] candidate_blocks.csv                     Rows:  1,204 | Columns:  20
[OK] conflict_analysis.csv                    Rows:    743 | Columns:  12
[OK] optimized_blocks.csv                     Rows:    258 | Columns:  17
[OK] baseline_comparison.csv                  Rows:    178 | Columns:  31
[OK] railwise_final_recommendations.csv       Rows:  6,000 | Columns:  19

--------------------------------------------------------------------------------
FINAL RECOMMENDATION CHECK
--------------------------------------------------------------------------------
Rows                 : 6,000
Unique block requests: 6,000
Duplicate IDs        : 0
Missing IDs          : 0

Request status:
request_status
NO FEASIBLE BLOCK    5251
DEFERRED              491
OPTIMIZED             258
Name: count, dtype: int64

--------------------------------------------------------------------------------
OPTI

,Output,File,Exists,Rows,Columns
0,Asset Risk Predictions,asset_risk_predictions.csv,YES,14400,13
1,Candidate Blocks,candidate_blocks.csv,YES,1204,20
2,Conflict Analysis,conflict_analysis.csv,YES,743,12
3,Optimized Blocks,optimized_blocks.csv,YES,258,17
4,Baseline Comparison,baseline_comparison.csv,YES,178,31
5,Final Recommendations,railwise_final_recommendations.csv,YES,6000,19
